# Modelos Multimodales
### Máster en IA y Computación Cuántica Aplicada a los Mercados Financieros (MIAX)

---

- **Autor:** Pablo Hernández Cámara
- **Contacto:** [pablo.hernandez-camara@uv.es](mailto:pablo.hernandez-camara@uv.es)
- **Notebook:** CLAP, conectando audio y texto

En los notebooks anteriores hemos trabajado con distintas modalidades: texto, imagen, vídeo y audio. Ahora damos un paso más: **relacionar audio y texto dentro de un mismo espacio de representación**.


Necesitamos `datasets` para cargar el ejemplo de audio desde Hugging Face y `transformers` para utilizar el modelo y su procesador.


In [1]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5


# Cargar datos

Vamos a utilizar un ejemplo pequeño procedente de **ESC-50**, un conjunto de datos de sonidos ambientales. En lugar de descargar manualmente un fichero, `datasets` nos permite cargarlo directamente. El elemento que nos interesa es el array de muestras de audio (`array`), es decir, la representación numérica de la señal.

In [ ]:
from datasets import load_dataset
from transformers import AutoProcessor, ClapModel

In [ ]:
dataset = load_dataset("hf-internal-testing/ashraq-esc50-1-dog-example")
audio_sample = dataset["train"]["audio"][0]["array"]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/451 [00:00<?, ?B/s]

data/train-00000-of-00001-8684bae6f519e8(…): reconstructing file:   0%|          |  0.00B / 50.2kB            

data/train-00000-of-00001-8684bae6f519e8(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
dataset["train"]["audio"][0]

In [ ]:
audio_sample

array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)

## Cargar el modelo

[CLAP](https://github.com/LAION-AI/CLAP) (*Contrastive Language-Audio Pretraining*) sigue una idea similar a la de **CLIP**, pero sustituyendo la pareja imagen-texto por **audio-texto**.

La pregunta que queremos responder es sencilla: **Dado un audio y varias descripciones de texto, ¿con cuál encaja mejor el audio?**

Utilizaremos `laion/clap-htsat-unfused`. Necesitamos dos componentes:

- **`ClapModel`**: contiene el modelo que transforma audio y texto en representaciones comparables.
- **`AutoProcessor`**: prepara cada modalidad en el formato que espera el modelo.

La idea conceptual, al igual que en CLIP, es:

**audio → representación de audio**  
**texto → representación de texto**  
**representaciones → medida de compatibilidad**

In [ ]:
model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
processor = AutoProcessor.from_pretrained("laion/clap-htsat-unfused")

config.json:   0%|          | 0.00/5.39k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  615MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  614MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Vamos a presentar al modelo un único audio y dos posibles descripciones:

1. `"Sound of a dog"`
2. `"Sound of a vacuum cleaner"`

El modelo calcula qué tan compatibles son el audio y cada texto.

Esto recuerda mucho a CLIP:

| CLIP | CLAP |
|---|---|
| Imagen ↔ texto | Audio ↔ texto |
| Representación visual | Representación de audio |
| Representación textual | Representación textual |
| Compatibilidad imagen-texto | Compatibilidad audio-texto |

Por tanto, podemos utilizar lenguaje natural para **consultar o clasificar un audio sin entrenar específicamente un clasificador para cada categoría**.

In [ ]:
input_text = ["Sound of a dog", "Sound of vaccum cleaner"]

inputs = processor(text=input_text, audio=audio_sample, return_tensors="pt", padding=True)

[transformers] It is strongly recommended to pass the `sampling_rate` argument to `ClapFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


In [ ]:
outputs = model(**inputs)
logits_per_audio = outputs.logits_per_audio
probs = logits_per_audio.softmax(dim=-1)

In [ ]:
probs

tensor([[0.7456, 0.0043, 0.2501]], grad_fn=<SoftmaxBackward0>)

Tenemos una probabilidad para cada texto candidato. No debemos interpretar este valor como una **probabilidad estadística absoluta de que el audio “sea” un perro**. Es más útil entenderlo como una distribución de preferencia del modelo entre las opciones que nosotros le hemos proporcionado.

Por ejemplo, si comparamos:

- `"Sound of a dog"`
- `"Sound of a vacuum cleaner"`

el modelo puede asignar más masa de probabilidad a una de ellas porque considera que encaja mejor con el audio.

Ahora mantenemos el mismo audio, pero cambiamos las alternativas:

- `"Sound of a big dog"`
- `"Sound of a small dog"`

Este experimento es especialmente interesante porque obliga a preguntarnos: ¿El audio contiene realmente información suficiente para distinguir entre un perro grande y uno pequeño?

Que el modelo produzca una preferencia **no significa que la característica esté objetivamente presente en el audio**.

In [ ]:
input_text = ["Sound of a big dog", "Sound of a small dog"]

inputs = processor(text=input_text, audio=audio_sample, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_audio = outputs.logits_per_audio
probs = logits_per_audio.softmax(dim=-1)

[transformers] It is strongly recommended to pass the `sampling_rate` argument to `ClapFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


In [ ]:
probs

tensor([[0.7711, 0.2289]], grad_fn=<SoftmaxBackward0>)